### pyAPES MLM benchmarking at DK-Sor (Soroe Beech Forest)



In [1]:
# setting path
import sys
#sys.path.append('c:\\Repositories\\pyAPES_main')
import os
from dotenv import load_dotenv

load_dotenv()
pyAPES_main_folder = os.getenv('pyAPES_main_folder')

sys.path.append(pyAPES_main_folder)
os.chdir(pyAPES_main_folder)  # set working directory so relative file paths resolve correctly
#print(sys.path)

# force iPython re-import modules at each call
%load_ext autoreload
%autoreload 2


### Import modules

In [16]:
# function to read forcing data. See 'forcing/forcing_info.txt' for model forcing variable names and units!


from pyAPES.pyAPES_MLM import driver

# import parameter dictionaries for Soroe
from pyAPES.parameters.mlm_parameters_Soroe import gpara, cpara, spara # model configuration, canopy parameters, soil parameters
from pyAPES.utils.iotools import read_forcing, read_results,  read_data

# python packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

eps = 1e-16

### Read forcing data and compile input for driver


In [17]:
forcing = read_forcing(
    forcing_file=gpara['forc_filename'],
    start_time=gpara['start_time'],
    end_time=gpara['end_time'],
    dt=gpara['dt']
)

params = {
    'general': gpara,   # model configuration
    'canopy': cpara,    # planttype, micromet, canopy, bottomlayer parameters
    'soil': spara,      # soil heat and water flow parameters
    'forcing': forcing  # forging data
}

In [18]:
cpara['radiation']['SWmodel']

'ZHAOQUALLS'

### Run the model

In [5]:
resultfile, Model = driver(parameters=params,
                           create_ncf=True,
                           result_file= 'Soroe_test.nc'
                          )

INFO pyAPES.pyAPES_MLM driver pyAPES_MLM simulation started. Number of simulations: 1
INFO pyAPES.soil.water __init__ Water balance in soil solved using: RICHARDS EQUATION & no lateral drainage
INFO pyAPES.soil.heat __init__ Soil heat balance solved.
INFO pyAPES.canopy.mlm_canopy __init__ Eflow: True, WMA: False, Ebal: True
INFO pyAPES.microclimate.radiation __init__ Shortwave radiation model: ZHAOQUALLS
INFO pyAPES.microclimate.radiation __init__ Longwave radiation model: ZHAOQUALLS
c:\Repositories\pyAPES_main\pyAPES\bottomlayer\organiclayer.py:1233: RuntimeWarning: invalid value encountered in scalar power
  Psi = -1e-2 / alfa*(s**(1.0 / m) - 1.0)**(1.0 / n)  # m
INFO pyAPES.canopy.forestfloor __init__ Forestfloor has 2 bottomlayer types
INFO pyAPES.pyAPES_MLM driver Running simulation number (start time 2026-03-18 14:30): 0
INFO pyAPES.pyAPES_MLM run Running simulation 0


0%.. 10%.. 20%.. 30%.. 40%.. 50%.. 60%.. 70%.. 80%.. 90%.. 

INFO pyAPES.pyAPES_MLM run Finished simulation 0, running time 147.90 seconds
INFO pyAPES.pyAPES_MLM driver Running time 147.90 seconds


100%


INFO pyAPES.pyAPES_MLM driver Ready! Results are in: results/Soroe_test.nc


### Read model results

In [6]:
# read simulation results to xarray dataset
results = read_results(os.path.join(pyAPES_main_folder, resultfile))

datafile = r'forcing\Soroe\DK-Sor_EC_2015-2020.dat'
# read observations from DK-Sor
data = read_data(os.path.join(pyAPES_main_folder, datafile), start_time=gpara['start_time'], end_time=gpara['end_time'])


c:\\Repositories\\pyAPES_main\results/Soroe_test.nc


In [ ]:
print(results)

# print list of all variables with their dimensions:
vars = list(results.data_vars)
for v in vars:
    print(f"{v}: {results[v].dims}")

### Plot some model results and compare with observations


In [7]:
sim = 0  # in this demo, we have only one simulation (i.e. only one parameter set was used)

# grid variables for plotting
t = results.date  # time
zc = results.canopy_z  # height above ground [m]
zs = results.soil_z  # depth of soil; shown negative [m]

In [ ]:
plt.plot(t, results.canopy_LAI)

### Soil temperature and moisture
- computed using *pyAPES.soil* submodels 'Water' and 'Heat'
- compare with measurements at similar depths (to be done - now compare with Ts and SWC at 5cm depth)

In [ ]:
%matplotlib qt
var = ['soil_temperature', 'soil_volumetric_liquid_water_content']

lyrs = [0, 6, 25, 35] # layers
#depths = np.array2string(np.asarray(zs[lyrs]), precision=1, separator=', ')
depths = ['{:.2f} m'.format(k) for k in zs[lyrs]]

fig, ax = plt.subplots(len(var), 1, figsize=(10,7))

k = 0
ax[0].plot(t, data['TS_F_MDS_1'], 'ko', markersize=3, alpha=0.1)
ax[0].plot(t, data['TS_F_MDS_2'], 'ko', markersize=3, alpha=0.2)
ax[0].plot(t, data['TS_F_MDS_3'], 'ko', markersize=3, alpha=0.4)
ax[0].plot(t, data['TS_F_MDS_4'], 'ko', markersize=3, alpha=0.6)

ax[1].plot(t, 1e-2 * data['SWC_F_MDS_1'], 'ko', markersize=3, alpha=0.1)
ax[1].plot(t, 1e-2 * data['SWC_F_MDS_2'], 'ko', markersize=3, alpha=0.2)
ax[1].plot(t, 1e-2 * data['SWC_F_MDS_3'], 'ko', markersize=3, alpha=0.4)
ax[1].plot(t, 1e-2 * data['SWC_F_MDS_4'], 'ko', markersize=3, alpha=0.6)
for v in var:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)
    k += 1

# # vertical profile at last timestep
# fig, ax = plt.subplots(1, 2) #figsize=(10,5))

# k = 0
# for v in var:
#     ax[k].plot(results[v][-1,sim,:], zs)
#     ax[k].set_xlabel(results[v].attrs['units'])
#     ax[0].set_ylabel('depth (m)')
#     k += 1

### Ecosystem-scale fluxes

- ecosystem - atm. fluxes represent the integrated sinks / sources in soil (soil-module), forestfloor (bottomlayer-module) and vegetation (planttype&canopy -modules)
- comparable to ecosystem - atmosphere exchange
- affected by current model forcing and sub-model instance state (e.g. Planttype and Canopy LAI, phenology, soil temperature, moisture etc.)


In [ ]:
data.columns

In [8]:
%matplotlib qt
# var = ['forcing_air_temperature', 'forcing_par', 'canopy_NEE', 'canopy_GPP', 
#        'canopy_Reco', 'canopy_Rnet', 'canopy_SH', 'canopy_LE', 'ffloor_ground_heat']

var = ['canopy_NEE', 'canopy_GPP', 'canopy_Reco']
var2 = ['canopy_Rnet', 'canopy_SH', 'canopy_LE', 'ffloor_ground_heat']


# # temperature & par
# ax[0].plot(t, results[var[0]][:,sim], 'k-', label=var[0])
# ax[0].set_ylabel(results[var[0]].attrs['units'])
# ax[0].tick_params(axis='x', labelrotation = 20)
# axb = ax[0].twinx()

# axb.plot(t, results[var[1]][:,sim], 'r-', label=var[1]) # Par
# axb.set_ylabel(results[var[1]].attrs['units'])

fig, ax = plt.subplots(3, 1, figsize=(8,10), sharex=True)
# CO2 fluxes
ax[0].plot(t, data['NEE_VUT_REF'], 'k.-', alpha=0.3)
ax[1].plot(t, data['GPP_NT_VUT_REF'], 'k.-', alpha=0.3)
ax[1].plot(t, data['GPP_DT_VUT_REF'], 'r.-', alpha=0.3)
ax[2].plot(t, data['RECO_NT_VUT_REF'], 'k.-', alpha=0.3)
ax[2].plot(t, data['RECO_DT_VUT_REF'], 'r.-', alpha=0.3)
n = 0
for v in var:
    ax[n].plot(t, results[v][:,sim], label=v)

    ax[n].set_ylabel('umol m-2 s-1')
    ax[n].tick_params(axis='x', labelrotation = 20)
    ax[n].legend(fontsize=8)
    n +=1

fig, ax = plt.subplots(5, 1, figsize=(8,10), sharex=True)
# energy fluxes
ax[1].plot(t, data['NETRAD'], 'k.-', alpha=0.3)
ax[2].plot(t, data['H_F_MDS'], 'k.-', alpha=0.3)
ax[3].plot(t, data['LE_F_MDS'], 'k.-', alpha=0.3)
ax[4].plot(t, data['G_F_MDS'], 'k.-', alpha=0.3)

n = 1
for v in var2:
    ax[0].plot(t, results[v][:,sim], label=v)
    ax[n].plot(t, results[v][:,sim], label=v)

    ax[n].set_ylabel('W m-2')
    ax[n].tick_params(axis='x', labelrotation = 20)
    ax[n].legend(fontsize=8)
    n += 1


In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(8,10), sharex=True)
# CO2 fluxes
ax[0].plot(t, data['GPP_NT_VUT_REF'], 'k.-', alpha=0.3); ax[0].set_ylabel('GPP (umolm-2s-1)')
ax[0].plot(t, results['canopy_GPP'][:,sim], label='GPP')

ax[1].plot(t, data['RECO_NT_VUT_REF'], 'k.-', alpha=0.3); ax[1].set_ylabel('RECO (umolm-2s-1)')
ax[1].plot(t, results['canopy_Reco'][:,sim], label='GPP')
ax[2].plot(t, data['TS_F_MDS_1'], 'k.-', alpha=0.3); ax[2].set_ylabel('T (degC)')
ax[3].plot(t, 1e-2*data['SWC_F_MDS_2'], 'k.-', alpha=0.3); ax[3].set_ylabel('SWC (m3m-3)')

ax[2].plot(t, results['soil_temperature'][:,sim,lyrs], label=depths)
ax[2].legend(fontsize=8)

var3 = ['soil_volumetric_liquid_water_content']
k = 3
for v in var3:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)



In [ ]:
print(data.columns)

### Ecosystem radiation balance

- net radiation (Rnet) is the sum of net shortwave (SWnet = incoming - reflected) and net longwave (LWnet = incoming - emitted) radiation at canopy top
- computed via models in pyAPES.microclimate.radiation, called iteratively from pyAPES.canopy.mlm_canopy to account for canopy structure and leaf & forest floor temperature
- ecosystem albedo can be computed from radiation profiles at uppermost grid point

In [ ]:
# net radiation components at canopy top
var = ['canopy_Rnet','canopy_SWnet', 'canopy_LWnet']
profs = ['canopy_par_down', 'canopy_par_up', 'canopy_nir_down','canopy_nir_up','canopy_lw_down','canopy_lw_up']

fig, ax = plt.subplots(3, 1, figsize=(8,12), sharex=True)

for v in var:
    ax[0].plot(t, results[v][:, sim], label=v)
ax[0].set_ylabel('W m-2')
ax[0].tick_params(axis='x', labelrotation = 20)
ax[0].legend(fontsize=8)    

# lets plot also partitioning of canopy_SWnet and canopy_LWnet.
# -1 is the index of uppermost gridpoint

for v in ['canopy_par_down', 'canopy_nir_down','canopy_lw_down']: # downward
    ax[1].plot(t, results[v][:,sim,-1], '-', label=v)
for v in ['canopy_par_up', 'canopy_nir_up','canopy_lw_up']: # upward
    ax[1].plot(t, results[v][:,sim,-1], '--', label=v)
ax[1].set_ylabel('W m-2')
ax[1].tick_params(axis='x', labelrotation = 20)
ax[1].legend(fontsize=8)

# canopy albedo
eps = 1e-16

# fraction of par on total SW
f_par = results['canopy_par_down'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + results['canopy_nir_down'][:,sim,-1] + eps)

alb_par = results['canopy_par_up'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + eps)
alb_nir = results['canopy_nir_up'][:,sim,-1] / (results['canopy_nir_down'][:,sim,-1] + eps)
alb_sw = f_par * alb_par + (1 - f_par) * alb_nir
alb_sw = np.maximum(0, np.minimum(1.0, alb_sw))

ax[2].plot(t, alb_sw, '-', label='SW albedo')
ax[2].plot(t, alb_par, '-', label='Par albedo')
ax[2].plot(t, alb_nir, '-', label='Nir albedo')

ax[2].tick_params(axis='x', labelrotation = 20)
ax[2].legend(fontsize=8)

In [ ]:
# Explore temperature-related variables in results
temp_vars = [v for v in results.data_vars if 'temp' in v.lower() or 'leaf' in v.lower()]
print("Temperature-related variables:")
for v in temp_vars:
    print(f"  {v}: dims={results[v].dims}, shape={results[v].shape}")

In [ ]:
%matplotlib qt
import numpy as np

# canopy height h = top of planttype[0] LAD profile (highest z where lad > 0)
pt0_lad = list(cpara['planttypes'].values())[0]['lad']
h = float(zc.values[pt0_lad > 0].max())

# find canopy indices closest to z/h = 0.2, 0.6, 1.0
target_ratios = [0.2, 0.6, 1.0]
labels = ['z/h = 0.2 (bottom)', 'z/h = 0.6 (middle)', 'z/h = 1.0 (top)']
colors = ['tab:blue', 'tab:orange', 'tab:green']

idx = [int(np.argmin(np.abs(zc.values - r * h))) for r in target_ratios]
actual_z = [float(zc[i]) for i in idx]
print(f"Canopy height h = {h:.1f} m  (planttype[0]: '{list(cpara['planttypes'].keys())[0]}')")
for r, i, z in zip(target_ratios, idx, actual_z):
    print(f"  z/h = {r} -> index {i}, z = {z:.2f} m")

# compute Tleaf - Tair at each height
Tair = results['canopy_temperature'][:, sim, :]
dT = results['canopy_Tleaf'][:, sim, :] - Tair

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
fig.suptitle('Leaf - air temperature difference (Tleaf - Tair)', fontsize=13)

for ax, i, label, color, z in zip(axes, idx, labels, colors, actual_z):
    ax.plot(t, dT[:, i], color=color, linewidth=0.7, label=f'dT  {label}  (z = {z:.1f} m)')
    ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_ylabel('Tleaf - Tair (deg C)', color=color)
    ax.tick_params(axis='y', labelcolor=color)
    ax.legend(fontsize=9, loc='upper left')
    ax.tick_params(axis='x', labelrotation=20)

    ax2 = ax.twinx()
    ax2.plot(t, Tair[:, i], color='gray', linewidth=0.7, alpha=0.7, label=f'Tair  {label}')
    ax2.set_ylabel('Tair (deg C)', color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')
    ax2.legend(fontsize=9, loc='upper right')

axes[-1].set_xlabel('Date')
fig.tight_layout()
plt.show()


### Microclimatic gradients within the canopy

- sub-models pyAPES.microclimate.Micromet & pyAPES.radiation.Radiation
- mean wind speed & friction velocity (momentum flux) profiles depend on momentum absorption in the canopy, which is proportional to leaf-area density (*lad*) profile. Canopy *lad* is sum of *lad* of each PlantType instance
- SW absorption depends on *lad* and leaf optical properties
- air-space scalar profiles (*T*, *H2O* & *CO2*) in steady-state with respective sink/source profile using 1st-order closure (K-theory)

Let's compute average profiles for daytime hours 10:00 -- 16:00 under conditions when canopy is dry


In [9]:
results.close()